In [ ]:
import pandas as pd
human_Percentages_df=pd.read_csv("../human_predictions/output_all.csv")
llm_Percentages_df=pd.read_csv("../llm_predictions/result_vwp50_scene_table.csv")

llm_Percentages_df=llm_Percentages_df.rename(columns={"Item":"stimuliId","Condition":"condition","Object":"obj"})
llm_Percentages_df["stimuliId"] = llm_Percentages_df["stimuliId"].astype(int)

#remove strange star symbols that come up in strings (even though we like stars they dont help here, sry Fei //Jonathan ^^)
llm_Percentages_df["Object"] = (
    llm_Percentages_df["Object"]
    .str.replace(r"^[^\w]+", "", regex=True)
    .str.strip()
)

#fixing different denotations of condition:
llm_Percentages_df["condition"] = llm_Percentages_df["condition"].replace("Restrictive", "restrictive")
llm_Percentages_df["condition"] = llm_Percentages_df["condition"].replace("Non-restr.", "non-restrictive")
llm_Percentages_df

,stimuliId,condition,Verb,Object,Surprisal (bits),P_norm,Rank,ΔS (bits)
0,1,restrictive,eat,★ cake,5.961,0.9734,1,6.482
1,1,restrictive,eat,ball,11.156,0.0266,2,-6.202
2,1,restrictive,eat,toy car,20.215,0.0000,3,-5.087
3,1,restrictive,eat,toy train,22.952,0.0000,4,-6.074
4,1,non-restrictive,move,ball,4.953,0.9934,1,-6.202
...,...,...,...,...,...,...,...,...
463,50,non-restrictive,pick up,scissors,11.491,0.5611,1,-5.245
464,50,non-restrictive,pick up,envelope,12.000,0.3941,2,-0.571
465,50,non-restrictive,pick up,★ stamp,15.759,0.0291,3,0.876
466,50,non-restrictive,pick up,sticker,16.894,0.0133,4,1.987


In [16]:
#TODO: investigate human percentages for trials where all are zero
human_Percentages_df

,stimuliId,partID,condition,obj,percent
0,6,1,non-restrictive,weighing-scales,0.000000
1,6,1,non-restrictive,jug,0.000000
2,6,1,non-restrictive,mushrooms,0.000000
3,6,1,non-restrictive,knife,0.000000
4,39,1,restrictive,shampoo,0.000000
...,...,...,...,...,...
995,41,5,restrictive,orange,0.134615
996,45,5,restrictive,laddle,0.000000
997,45,5,restrictive,peeler,0.000000
998,45,5,restrictive,pot,0.000000


In [ ]:
#first before comparing we need to aggregate. not over all participants of a trial 
# but over all participants with the same contidion (4-restrictive 4 non restrictive) however some may not be present yet...
avg_df_human = (
    human_Percentages_df
    .groupby(["stimuliId", "condition", "obj"])
    .agg(
        avg_percent=("percent", "mean"),
        n_participants=("partID", "nunique")
    )
    .reset_index()
)
avg_df_human.to_csv("01_intermediate_human_aggregation.csv")
avg_df_human

,stimuliId,condition,obj,avg_percent,n_participants
0,0,non-restrictive,ball,0.0,3
1,0,non-restrictive,cake,0.0,3
2,0,non-restrictive,toy-car,0.0,3
3,0,non-restrictive,toy-train,0.0,3
4,0,restrictive,ball,0.0,2
...,...,...,...,...,...
403,49,restrictive,pouch,0.0,1
404,49,restrictive,pump,0.0,1
405,49,restrictive,rubberband,0.0,2
406,49,restrictive,scissors,0.0,2


In [ ]:
#check after avergaing over all participants how many stimuli got only 0 percentages...

In [ ]:
#now we want to create a shared dataframe that maps the percentages contains both human and llm prediction per word per condition

shared_df=avg_df_human.merge(llm_Percentages_df, on=["stimuliId","condition","obj"])
shared_df

,stimuliId,condition,obj,avg_percent,n_participants,Verb,Object,Surprisal (bits),P_norm,Rank,ΔS (bits)
0,1,non-restrictive,chair,0.0,2,move,ball,4.953,0.9934,1,-6.202
1,1,non-restrictive,chair,0.0,2,move,★ cake,12.443,0.0055,2,6.482
2,1,non-restrictive,chair,0.0,2,move,toy car,15.128,0.0009,3,-5.087
3,1,non-restrictive,chair,0.0,2,move,toy train,16.878,0.0003,4,-6.074
4,1,non-restrictive,cheese,0.0,2,move,ball,4.953,0.9934,1,-6.202
...,...,...,...,...,...,...,...,...,...,...,...
1863,49,restrictive,stamp,0.0,2,inflate,★ balloon,6.902,0.8776,1,5.998
1864,49,restrictive,stamp,0.0,2,inflate,flag,11.089,0.0482,2,-2.689
1865,49,restrictive,stamp,0.0,2,inflate,pump,11.382,0.0393,3,4.961
1866,49,restrictive,stamp,0.0,2,inflate,pouch,12.548,0.0175,4,-0.128
